In [1]:
import sys
sys.path.append("../src")

from eda_utils import (
    missing_values,
)
from prepro_utils import (
    classify_source,
    group_missing_by_source,
    impute_missing,
)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


In [3]:
#path setup
PROCESSED = Path("../data/processed")
# Loadinf the data
df_train = pd.read_parquet(PROCESSED / "feature_matrix_train.parquet")
df_test  = pd.read_parquet(PROCESSED / "feature_matrix_test.parquet")

print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")

Train shape: (307511, 249)
Test shape:  (48744, 248)


In [4]:
# checking the sentinel if was changed or not for train
df_checking_train = df_train[["ORGANIZATION_TYPE","CODE_GENDER","NAME_FAMILY_STATUS"]]
for i in df_checking_train:
    print(f"\n{'─'*50}")
    print(f"{i} — {df_train[i].nunique()} unique values")
    print(df_train[i].value_counts(dropna=False))


──────────────────────────────────────────────────
ORGANIZATION_TYPE — 58 unique values
ORGANIZATION_TYPE
Business Entity Type 3    67992
XNA                       55374
Self-employed             38412
Other                     16683
Medicine                  11193
Business Entity Type 2    10553
Government                10404
School                     8893
Trade: type 7              7831
Kindergarten               6880
Construction               6721
Business Entity Type 1     5984
Transport: type 4          5398
Trade: type 3              3492
Industry: type 9           3368
Industry: type 3           3278
Security                   3247
Housing                    2958
Industry: type 11          2704
Military                   2634
Bank                       2507
Agriculture                2454
Police                     2341
Transport: type 2          2204
Postal                     2157
Security Ministries        1974
Trade: type 2              1900
Restaurant                 18

In [5]:
# checking the sentinel if was changed or not for test
df_checking_test = df_test[["ORGANIZATION_TYPE","CODE_GENDER","NAME_FAMILY_STATUS"]]
for i in df_checking_test:
    print(f"\n{'─'*50}")
    print(f"{i} — {df_test[i].nunique()} unique values")
    print(df_test[i].value_counts(dropna=False))


──────────────────────────────────────────────────
ORGANIZATION_TYPE — 58 unique values
ORGANIZATION_TYPE
Business Entity Type 3    10840
XNA                        9274
Self-employed              5920
Other                      2707
Medicine                   1716
Government                 1508
Business Entity Type 2     1479
Trade: type 7              1303
School                     1287
Construction               1039
Kindergarten               1038
Business Entity Type 1      887
Transport: type 4           884
Trade: type 3               578
Military                    530
Industry: type 9            499
Industry: type 3            489
Security                    472
Transport: type 2           448
Police                      441
Housing                     435
Industry: type 11           416
Bank                        374
Security Ministries         341
Services                    302
Postal                      294
Agriculture                 292
Restaurant                  2

In [6]:
# value counts for the bellow columns, and confirm the counts match what we found in Phase 2 EDA
df_checking_flag = df_train[["FLAG_NO_BUREAU_HISTORY", "FLAG_NO_PREV_APPLICATION","FLAG_HAS_CREDIT_CARD"]]

for clm in df_checking_flag:
    print(f"\n{'─'*50}")
    print(f"{clm} - {df_train[clm].nunique()} unique values")
    print(df_train[clm].value_counts(dropna=False))



──────────────────────────────────────────────────
FLAG_NO_BUREAU_HISTORY - 2 unique values
FLAG_NO_BUREAU_HISTORY
0    263491
1     44020
Name: count, dtype: int64

──────────────────────────────────────────────────
FLAG_NO_PREV_APPLICATION - 2 unique values
FLAG_NO_PREV_APPLICATION
0    291057
1     16454
Name: count, dtype: int64

──────────────────────────────────────────────────
FLAG_HAS_CREDIT_CARD - 2 unique values
FLAG_HAS_CREDIT_CARD
0    229577
1     77934
Name: count, dtype: int64


In [7]:
# checking all the missing values from the train df
missing_pct = (df_train.isna().sum() / len(df_train) * 100)
df_missing_grouped = group_missing_by_source(missing_pct)
    

                      n_columns  n_with_missing  mean_missing_pct  \
source                                                              
credit_card                  23              22            72.167   
bureau_balance               11              11            70.007   
application_train           138              90            23.206   
bureau                       23              22            14.937   
pos_cash                     11              11             6.682   
installments                 17              17             5.888   
previous_application         26              25             5.174   

                      max_missing_pct  
source                                 
credit_card                    83.318  
bureau_balance                 70.007  
application_train              69.872  
bureau                         40.202  
pos_cash                        6.691  
installments                    5.891  
previous_application            5.681  


In [8]:
# Bureau anomaly investigation (why max 40.2%?)
bureau_missing = df_missing_grouped[df_missing_grouped["source"] == "bureau"]
bureau_missing = bureau_missing.sort_values("missing_pct", ascending=False)
print(bureau_missing.to_string(index=False))

                   feature  missing_pct source
MAX_AMT_CREDIT_MAX_OVERDUE       40.202 bureau
  MEAN_AMT_CREDIT_SUM_DEBT       16.708 bureau
      DEBT_TO_CREDIT_RATIO       14.667 bureau
       MEAN_AMT_CREDIT_SUM       14.315 bureau
SUM_AMT_CREDIT_SUM_OVERDUE       14.315 bureau
     RATIO_ACTIVE_TO_TOTAL       14.315 bureau
      COUNT_CREDIT_PROLONG       14.315 bureau
   DAYS_SINCE_FIRST_CREDIT       14.315 bureau
    DAYS_SINCE_LAST_CREDIT       14.315 bureau
   SUM_AMT_CREDIT_SUM_DEBT       14.315 bureau
        SUM_AMT_CREDIT_SUM       14.315 bureau
        COUNT_BUREAU_TOTAL       14.315 bureau
       COUNT_BUREAU_ACTIVE       14.315 bureau
    MAX_CREDIT_DAY_OVERDUE       14.315 bureau
       FLAG_EVER_PROLONGED       14.315 bureau
         FLAG_HAS_MORTGAGE       14.315 bureau
        FLAG_HAS_MICROLOAN       14.315 bureau
         FLAG_EVER_OVERDUE       14.315 bureau
  FLAG_HAS_BAD_DEBT_BUREAU       14.315 bureau
         COUNT_BUREAU_SOLD       14.315 bureau
     COUNT_BU

In [9]:
# Application train columns, split into likely buckets
app_missing = df_missing_grouped[
    (df_missing_grouped["source"] == "application_train") &
    (df_missing_grouped["missing_pct"] > 0)
].sort_values("missing_pct", ascending=False)

print(f"Total application_train columns with missing: {len(app_missing)}\n")
print(app_missing.to_string(index=False))

Total application_train columns with missing: 90

                        feature  missing_pct            source
                 COMMONAREA_AVG       69.872 application_train
                COMMONAREA_MEDI       69.872 application_train
                COMMONAREA_MODE       69.872 application_train
        NONLIVINGAPARTMENTS_AVG       69.433 application_train
       NONLIVINGAPARTMENTS_MEDI       69.433 application_train
       NONLIVINGAPARTMENTS_MODE       69.433 application_train
             FONDKAPREMONT_MODE       68.386 application_train
          LIVINGAPARTMENTS_MODE       68.355 application_train
           LIVINGAPARTMENTS_AVG       68.355 application_train
          LIVINGAPARTMENTS_MEDI       68.355 application_train
                 FLOORSMIN_MEDI       67.849 application_train
                  FLOORSMIN_AVG       67.849 application_train
                 FLOORSMIN_MODE       67.849 application_train
                    OWN_CAR_AGE       66.785 application_train
     

In [10]:
# Verify the child-table groups behave uniformly (all ~coverage-gap missing rates)
for source in ["credit_card", "bureau_balance", "pos_cash", "installments", "previous_application"]:
    subset = df_missing_grouped[
        (df_missing_grouped["source"] == source) &
        (df_missing_grouped["missing_pct"] > 0)
    ].sort_values("missing_pct", ascending=False)
    print(f"\n{'='*60}")
    print(f"  {source} — {len(subset)} columns with missing")
    print(f"{'='*60}")
    print(subset.to_string(index=False))


  credit_card — 22 columns with missing
                      feature  missing_pct      source
       CC_MEAN_PAYMENT_VS_MIN       83.318 credit_card
CC_ATM_TO_TOTAL_DRAWING_RATIO       82.827 credit_card
           CC_MAX_UTILISATION       74.940 credit_card
          CC_MEAN_UTILISATION       74.940 credit_card
         CC_FLAG_EVER_DPD_DEF       74.657 credit_card
CC_COUNT_MONTHS_WITH_DRAWINGS       74.657 credit_card
         CC_MEAN_AMT_DRAWINGS       74.657 credit_card
         CC_MEAN_CREDIT_LIMIT       74.657 credit_card
           CC_MAX_AMT_BALANCE       74.657 credit_card
          CC_MEAN_AMT_BALANCE       74.657 credit_card
          CC_COUNT_MONTHS_DPD       74.657 credit_card
                   CC_MAX_DPD       74.657 credit_card
      CC_FLAG_EVER_OVER_LIMIT       74.657 credit_card
             CC_FLAG_EVER_DPD       74.657 credit_card
       CC_FLAG_EVER_HIGH_UTIL       74.657 credit_card
  CC_COUNT_MONTHS_ATM_DRAWING       74.657 credit_card
     CC_FLAG_EVER_ATM_DR

In [11]:
df_train, train_medians = impute_missing(df_train)
df_test, _ = impute_missing(df_test, medians=train_medians)

In [12]:
missing_values(df_train)

🟢 Percent missing:
CC_MEAN_PAYMENT_VS_MIN          83.320
CC_ATM_TO_TOTAL_DRAWING_RATIO   82.830
CC_MAX_UTILISATION              74.940
CC_MEAN_UTILISATION             74.940
CC_MEAN_AMT_DRAWINGS            74.660
                                 ...  
MEAN_AMT_CREDIT                  5.420
REFUSAL_RATE                     5.350
APPROVAL_RATE                    5.350
DAYS_SINCE_LAST_DECISION         5.350
NAME_TYPE_SUITE                  0.420
Length: 95, dtype: float64

🟢 Number missing:
CC_MEAN_PAYMENT_VS_MIN           256213
CC_ATM_TO_TOTAL_DRAWING_RATIO    254701
CC_MAX_UTILISATION               230450
CC_MEAN_UTILISATION              230450
CC_MEAN_AMT_DRAWINGS             229577
                                  ...  
REFUSAL_RATE                      16454
APPROVAL_RATE                     16454
NAME_TYPE_SUITE                    1292
CODE_GENDER                           4
NAME_FAMILY_STATUS                    2
Length: 97, dtype: int64


In [13]:
print(f"SK_ID_CURR missing: {df_train['SK_ID_CURR'].isna().sum()}")
print(f"CC_COUNT_MONTHS_DPD missing: {df_train['CC_COUNT_MONTHS_DPD'].isna().sum()}")
print(f"EXT_SOURCE_2 missing: {df_train['EXT_SOURCE_2'].isna().sum()}")
print(f"EXT_SOURCE_1 missing: {df_train['EXT_SOURCE_1'].isna().sum()}")

SK_ID_CURR missing: 0
CC_COUNT_MONTHS_DPD missing: 0
EXT_SOURCE_2 missing: 0
EXT_SOURCE_1 missing: 173378
